# Fall Detection System — Full Edition (MediaPipe 0.10+)
### Acceleration Analysis + Sound Alert + CSV Log + Webcam/Video Support

---

## Why this version?
MediaPipe **0.10+** removed `mp.solutions`. This notebook uses the new **`mediapipe.tasks`** API.

## Concept pipeline
| Layer | Math concept | Applied as |
|---|---|---|
| **1** | Straight-line geometry — midpoint, Euclidean distance, dot-product angle | `landmark_point`, `torso_angle_from_vertical` |
| **2** | Acceleration = Δvelocity / Δt from frame timestamps | `DerivativeTracker` |
| **3** | Multi-condition fall gate (A AND B AND C) | `FallDetector.update()` |

## Steps
1. **Cell 1** — install packages + download the pose model (~7 MB, once only)
2. **Cell 2** — load all helpers and classes
3. **Cell 3** — set `SOURCE`, run to start, click Stop or interrupt kernel to stop
4. **Cell 4** — post-session chart from the saved CSV

In [1]:
# Cell 1 - Install and download model
# mediapipe 0.10+ uses the Tasks API — no mp.solutions needed
%pip install mediapipe opencv-python numpy matplotlib ipywidgets --quiet

import urllib.request, pathlib
MODEL_PATH = 'pose_landmarker_lite.task'
if not pathlib.Path(MODEL_PATH).exists():
    url = ('https://storage.googleapis.com/mediapipe-models/'
           'pose_landmarker/pose_landmarker_lite/float16/latest/'
           'pose_landmarker_lite.task')
    print('Downloading pose model (~7 MB)...')
    urllib.request.urlretrieve(url, MODEL_PATH)
    print('Download complete.')
else:
    print('Model file already present.')

import mediapipe as mp
print(f'mediapipe {mp.__version__} ready')

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


Note: you may need to restart the kernel to use updated packages.
Model file already present.
mediapipe 0.10.35 ready


In [2]:
import cv2

cap = cv2.VideoCapture(0)
print('Opened:', cap.isOpened())
print('Backend:', cap.getBackendName())
cap.release()

Opened: True
Backend: MSMF


In [3]:
# Cell 2 - Imports, helpers, classes
import cv2
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

import numpy as np
import math, time, csv, platform, subprocess, threading, datetime, collections

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from IPython.display import display
import ipywidgets as widgets

# Landmark index constants (same numbering as the old API)
# Full list: https://developers.google.com/mediapipe/solutions/vision/pose_landmarker
LEFT_SHOULDER  = 11
RIGHT_SHOULDER = 12
LEFT_HIP       = 23
RIGHT_HIP      = 24

POSE_CONNECTIONS = [
    (11,12),(11,13),(13,15),(12,14),(14,16),
    (11,23),(12,24),(23,24),
    (23,25),(25,27),(24,26),(26,28),
    (27,29),(29,31),(28,30),(30,32),
]


# ===== LAYER 1: Straight-line geometry =====

def landmark_point(lm, w, h):
    """Normalised landmark (x,y in [0,1]) to pixel coordinate."""
    return np.array([lm.x * w, lm.y * h], dtype=float)


def torso_angle_from_vertical(shoulder_mid, hip_mid):
    """Angle of torso vector from the vertical axis.
    torso_vec = shoulder_mid - hip_mid
    vertical  = [0, -1]  (upward; image y grows downward)
    cos(theta) = dot(torso_vec, vertical) / |torso_vec|
    0 deg = upright, 90 deg = lying flat (fall territory)
    """
    v  = shoulder_mid - hip_mid
    up = np.array([0.0, -1.0])
    cos_t = np.dot(v, up) / (np.linalg.norm(v) + 1e-9)
    return math.degrees(math.acos(np.clip(cos_t, -1.0, 1.0)))


def draw_skeleton(frame, landmarks, w, h):
    """Manual skeleton draw — replaces the old mp_draw.draw_landmarks."""
    pts = [landmark_point(lm, w, h) for lm in landmarks]
    for a, b in POSE_CONNECTIONS:
        if a < len(pts) and b < len(pts):
            cv2.line(frame, tuple(pts[a].astype(int)),
                     tuple(pts[b].astype(int)), (180, 180, 180), 1)
    for p in pts:
        cv2.circle(frame, tuple(p.astype(int)), 3, (255, 255, 255), -1)


# ===== LAYER 2: Acceleration from frame timestamps =====

class DerivativeTracker:
    """Finite-difference velocity and acceleration.
    velocity[t]     = (y[t] - y[t-1]) / dt    pixels/second
    acceleration[t] = (vel[t] - vel[t-1]) / dt pixels/second^2
    Positive values = downward motion (image y grows downward).
    """
    def __init__(self, window=6):
        self.positions  = collections.deque(maxlen=window)
        self.velocities = collections.deque(maxlen=window)

    def update(self, y, t):
        if self.positions:
            t0, y0 = self.positions[-1]
            dt = t - t0
            if dt > 0:
                self.velocities.append((t, (y - y0) / dt))
        self.positions.append((t, y))

    def velocity(self):
        return self.velocities[-1][1] if self.velocities else 0.0

    def acceleration(self):
        """Most recent vertical acceleration in px/s^2. Positive = downward."""
        if len(self.velocities) < 2:
            return 0.0
        t1, v1 = self.velocities[-2]
        t2, v2 = self.velocities[-1]
        dt = t2 - t1
        return (v2 - v1) / dt if dt > 0 else 0.0


# ===== LAYER 3: Fall decision =====

class FallDetector:
    """Three-condition AND gate.
    A: hip vertical acceleration > ACCEL_THRESHOLD  (rapid downward motion)
    B: torso angle from vertical > ANGLE_THRESHOLD  (body tilting or horizontal)
    C: hip y-position > HIP_HEIGHT_RATIO x frame_h  (person is low in frame)
    All three true simultaneously = FALL DETECTED.
    """
    ACCEL_THRESHOLD  = 800
    ANGLE_THRESHOLD  = 45
    HIP_HEIGHT_RATIO = 0.65
    COOLDOWN         = 5.0
    ALERT_DISPLAY    = 3.0

    def __init__(self):
        self.tracker      = DerivativeTracker()
        self.last_alert_t = 0.0
        self.alert_active = False
        self.alert_t      = 0.0
        self.fall_count   = 0

    def update(self, hip_mid, shoulder_mid, frame_h, t):
        self.tracker.update(hip_mid[1], t)
        accel       = self.tracker.acceleration()
        torso_angle = torso_angle_from_vertical(shoulder_mid, hip_mid)
        hip_ratio   = hip_mid[1] / frame_h

        cond_a = accel       > self.ACCEL_THRESHOLD
        cond_b = torso_angle > self.ANGLE_THRESHOLD
        cond_c = hip_ratio   > self.HIP_HEIGHT_RATIO

        if cond_a and cond_b and cond_c:
            if (t - self.last_alert_t) > self.COOLDOWN:
                self.fall_count  += 1
                self.last_alert_t = t
                self.alert_active = True
                self.alert_t      = t
                play_alert_sound()
                print(f'FALL #{self.fall_count}  t={t:.2f}s  '
                      f'accel={accel:.0f}  angle={torso_angle:.1f} deg')

        if self.alert_active and (t - self.alert_t) > self.ALERT_DISPLAY:
            self.alert_active = False

        return self.alert_active, {
            'accel': accel, 'torso_angle': torso_angle, 'hip_ratio': hip_ratio,
            'cond_a': cond_a, 'cond_b': cond_b, 'cond_c': cond_c,
            'fall_count': self.fall_count,
        }


# ===== SOUND ALERT =====

def play_alert_sound():
    """Non-blocking cross-platform alert beep."""
    def _play():
        try:
            s = platform.system()
            if s == 'Windows':
                import winsound
                winsound.Beep(880, 400)
                winsound.Beep(660, 400)
            elif s == 'Darwin':
                subprocess.run(['afplay', '/System/Library/Sounds/Sosumi.aiff'],
                               check=False)
            else:
                for snd in ['/usr/share/sounds/freedesktop/stereo/dialog-warning.oga',
                            '/usr/share/sounds/ubuntu/stereo/dialog-warning.ogg']:
                    import os
                    if os.path.exists(snd):
                        subprocess.run(['paplay', snd], check=False)
                        break
        except Exception as e:
            print(f'[sound] {e}')
    threading.Thread(target=_play, daemon=True).start()


# ===== CSV LOGGER =====

class CSVLogger:
    FIELDS = ['timestamp','fps','accel','torso_angle','hip_ratio',
              'cond_a','cond_b','cond_c','fall_detected','fall_count']

    def __init__(self, path):
        self.path = path
        self._f = open(path, 'w', newline='', encoding='utf-8')
        self._w = csv.DictWriter(self._f, fieldnames=self.FIELDS)
        self._w.writeheader()
        print(f'Logging to {path}')

    def write(self, t, fps, diag, is_fall):
        self._w.writerow({
            'timestamp': f'{t:.4f}', 'fps': f'{fps:.1f}',
            'accel': f'{diag["accel"]:+.2f}',
            'torso_angle': f'{diag["torso_angle"]:.2f}',
            'hip_ratio': f'{diag["hip_ratio"]:.4f}',
            'cond_a': int(diag['cond_a']), 'cond_b': int(diag['cond_b']),
            'cond_c': int(diag['cond_c']), 'fall_detected': int(is_fall),
            'fall_count': diag['fall_count'],
        })

    def close(self):
        self._f.flush()
        self._f.close()
        print(f'Log saved to {self.path}')


# ===== LIVE CHART =====

CHART_LEN = 120

def render_chart(chart_buf, fall_events):
    import io
    if len(chart_buf) < 2:
        return None
    ts     = [r['t'] for r in chart_buf]
    t0     = ts[0]
    xs     = [t - t0 for t in ts]
    accels = [r['accel'] for r in chart_buf]
    angles = [r['torso_angle'] for r in chart_buf]

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(5, 3.2),
                                    facecolor='#111', tight_layout=True)
    for ax in (ax1, ax2):
        ax.set_facecolor('#1a1a1a')
        ax.tick_params(colors='#888', labelsize=7)
        for sp in ax.spines.values():
            sp.set_color('#333')
        for fe in fall_events:
            xf = fe - t0
            if xs and xs[0] <= xf <= xs[-1]:
                ax.axvline(xf, color='red', alpha=0.6, linewidth=1)

    ax1.plot(xs, accels, color='#4fc3f7', linewidth=1)
    ax1.axhline(800, color='#ef5350', linewidth=0.8, linestyle='--')
    ax1.set_ylabel('Accel (px/s2)', color='#888', fontsize=7)
    ax1.set_title('Vertical Acceleration', color='#aaa', fontsize=8)

    ax2.plot(xs, angles, color='#aed581', linewidth=1)
    ax2.axhline(45, color='#ef5350', linewidth=0.8, linestyle='--')
    ax2.set_ylabel('Angle (deg)', color='#888', fontsize=7)
    ax2.set_title('Torso Angle from Vertical', color='#aaa', fontsize=8)
    ax2.set_xlabel('seconds', color='#888', fontsize=7)

    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=100, facecolor='#111')
    plt.close(fig)
    buf.seek(0)
    return buf.read()


# ===== HUD OVERLAY =====

def draw_hud(frame, diag, is_fall, fps):
    h, w = frame.shape[:2]
    ov = frame.copy()
    cv2.rectangle(ov, (0, 0), (295, h), (12, 12, 12), -1)
    cv2.addWeighted(ov, 0.58, frame, 0.42, 0, frame)

    ok  = lambda v: (60, 220, 100) if v else (80, 80, 80)
    dim = (130, 130, 130)
    hdr = (210, 210, 210)

    rows = [
        ('FALL DETECTION',                                            hdr, 0.54, 2),
        (f'FPS: {fps:.1f}   Falls: {diag["fall_count"]}',            dim, 0.40, 1),
        ('',                                                         None, 0,    0),
        ('-- LAYER 2: Acceleration --',                               dim, 0.37, 1),
        (f'Hip accel : {diag["accel"]:+.0f} px/s2', ok(diag['cond_a']),  0.42, 1),
        ('  threshold: 800 px/s2',                                   dim, 0.36, 1),
        ('',                                                         None, 0,    0),
        ('-- LAYER 1+3: Geometry --',                                 dim, 0.37, 1),
        (f'Torso ang : {diag["torso_angle"]:.1f} deg', ok(diag['cond_b']), 0.42, 1),
        (f'Hip height: {diag["hip_ratio"]*100:.1f}%',  ok(diag['cond_c']), 0.42, 1),
        ('',                                                         None, 0,    0),
        ('-- CONDITIONS (A AND B AND C) --',                          dim, 0.37, 1),
        (f'[A] accel  {"PASS" if diag["cond_a"] else "fail"}', ok(diag['cond_a']), 0.42, 1),
        (f'[B] angle  {"PASS" if diag["cond_b"] else "fail"}', ok(diag['cond_b']), 0.42, 1),
        (f'[C] height {"PASS" if diag["cond_c"] else "fail"}', ok(diag['cond_c']), 0.42, 1),
    ]
    y = 28
    for text, color, scale, thick in rows:
        if not text or scale == 0:
            y += 9
            continue
        cv2.putText(frame, text, (10, y),
                    cv2.FONT_HERSHEY_SIMPLEX, scale, color, thick, cv2.LINE_AA)
        y += int(scale * 46 + 3)

    if is_fall:
        bn = frame.copy()
        cv2.rectangle(bn, (0, 0), (w, h), (0, 0, 140), -1)
        cv2.addWeighted(bn, 0.25, frame, 0.75, 0, frame)
        bx1, by1, bx2, by2 = 305, h//2 - 52, w - 8, h//2 + 52
        cv2.rectangle(frame, (bx1, by1), (bx2, by2), (0, 0, 190), -1)
        cv2.rectangle(frame, (bx1, by1), (bx2, by2), (0, 0, 255), 2)
        cv2.putText(frame, '!! FALL DETECTED !!',
                    (bx1 + 14, h//2 - 12), cv2.FONT_HERSHEY_DUPLEX,
                    0.85, (255, 255, 255), 2, cv2.LINE_AA)
        cv2.putText(frame, 'Check on the person immediately',
                    (bx1 + 14, h//2 + 28), cv2.FONT_HERSHEY_SIMPLEX,
                    0.45, (220, 170, 170), 1, cv2.LINE_AA)


print('All helpers loaded. Run Cell 3 to start.')

All helpers loaded. Run Cell 3 to start.


In [4]:
# =====================================================
# Cell 3A - LIVE WEBCAM / VIDEO FILE
# =====================================================
# CONFIG — edit only these lines
SOURCE             = 'webcam'                    # 'webcam' or 'path/to/video.mp4'
CAMERA_INDEX       = 0                           # 0 = built-in, 1+ = external
DISPLAY_W          = 960
JPEG_QUALITY       = 80
CHART_UPDATE_EVERY = 8
MODEL_PATH         = 'pose_landmarker_lite.task' # downloaded in Cell 1

# MediaPipe Tasks API setup (0.10 compatible)
BaseOptions           = mp_python.BaseOptions
PoseLandmarker        = mp_vision.PoseLandmarker
PoseLandmarkerOptions = mp_vision.PoseLandmarkerOptions
VisionRunningMode     = mp_vision.RunningMode

options = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=VisionRunningMode.VIDEO,
    num_poses=1,
    min_pose_detection_confidence=0.6,
    min_pose_presence_confidence=0.6,
    min_tracking_confidence=0.5,
)

# Open video source
cap = cv2.VideoCapture(CAMERA_INDEX if SOURCE == 'webcam' else SOURCE)
if not cap.isOpened():
    raise RuntimeError(f'Cannot open source: {SOURCE!r}')
print(f'Source: {"Webcam " + str(CAMERA_INDEX) if SOURCE == "webcam" else SOURCE}')

# CSV logger
ts_str   = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
log_path = f'fall_log_{ts_str}.csv'
logger   = CSVLogger(log_path)

# Widgets
img_widget   = widgets.Image(format='jpeg', width=DISPLAY_W)
chart_widget = widgets.Image(format='png',  width=480)
stop_btn     = widgets.Button(description='Stop', button_style='danger',
                              layout=widgets.Layout(width='100px'))
status_lbl   = widgets.Label(value='Starting...')
fall_lbl     = widgets.Label(value='Falls: 0')

display(widgets.VBox([
    widgets.HBox([img_widget, chart_widget]),
    widgets.HBox([stop_btn, status_lbl, fall_lbl]),
]))

running = [True]
stop_btn.on_click(lambda _: running.__setitem__(0, False))

detector      = FallDetector()
chart_buf     = collections.deque(maxlen=CHART_LEN)
fall_events   = []
encode_params = [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY]
prev_t  = time.time()
fps_ema = 0.0
frame_i = 0

with PoseLandmarker.create_from_options(options) as landmarker:
    while running[0]:
        ok, frame = cap.read()
        if not ok:
            status_lbl.value = 'End of source.'
            break

        now     = time.time()
        fps_ema = 0.9 * fps_ema + 0.1 / max(now - prev_t, 1e-6)
        prev_t  = now
        frame_i += 1

        if DISPLAY_W:
            scale = DISPLAY_W / frame.shape[1]
            frame = cv2.resize(frame, (DISPLAY_W, int(frame.shape[0] * scale)))
        h, w = frame.shape[:2]

        # MediaPipe Tasks inference
        # detect_for_video needs a monotonically increasing timestamp in ms
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB,
                            data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        result = landmarker.detect_for_video(mp_image, int(now * 1000))

        is_fall = False
        diag    = dict(accel=0, torso_angle=0, hip_ratio=0,
                       cond_a=False, cond_b=False, cond_c=False, fall_count=0)

        if result.pose_landmarks:
            lms = result.pose_landmarks[0]   # first detected person

            # LAYER 1 - pixel coords and midpoints
            ls = landmark_point(lms[LEFT_SHOULDER],  w, h)
            rs = landmark_point(lms[RIGHT_SHOULDER], w, h)
            lh = landmark_point(lms[LEFT_HIP],       w, h)
            rh = landmark_point(lms[RIGHT_HIP],      w, h)

            shoulder_mid = (ls + rs) / 2    # midpoint = (p1 + p2) / 2
            hip_mid      = (lh + rh) / 2

            # LAYERS 2+3 - acceleration to fall decision
            prev_count = detector.fall_count
            is_fall, diag = detector.update(hip_mid, shoulder_mid, h, now)

            if detector.fall_count > prev_count:
                fall_events.append(now)
                fall_lbl.value = f'Falls: {detector.fall_count}'

            # Skeleton overlay
            draw_skeleton(frame, lms, w, h)
            # Torso axis (cyan)
            cv2.line(frame, tuple(hip_mid.astype(int)),
                     tuple(shoulder_mid.astype(int)), (100, 220, 255), 3)
            # Vertical reference line from hip (blue)
            cv2.line(frame, tuple(hip_mid.astype(int)),
                     (int(hip_mid[0]), int(hip_mid[1]) - 80), (60, 60, 200), 1)

        draw_hud(frame, diag, is_fall, fps_ema)
        logger.write(now, fps_ema, diag, is_fall)

        chart_buf.append({'t': now, 'accel': diag['accel'],
                          'torso_angle': diag['torso_angle']})

        _, buf = cv2.imencode('.jpg', frame, encode_params)
        img_widget.value = buf.tobytes()

        if frame_i % CHART_UPDATE_EVERY == 0:
            png = render_chart(chart_buf, fall_events)
            if png:
                chart_widget.value = png

        status_lbl.value = (
            f'fps={fps_ema:.1f}  accel={diag["accel"]:+.0f}  '
            f'angle={diag["torso_angle"]:.1f} deg  '
            f'hip={diag["hip_ratio"]*100:.1f}%  '
            f'{"FALL" if is_fall else "ok"}'
        )

cap.release()
logger.close()
status_lbl.value = f'Stopped. Log: {log_path}'
print(f'Total falls: {detector.fall_count}  |  Log: {log_path}')

Source: Webcam 0
Logging to fall_log_20260524_142754.csv


KeyboardInterrupt: 

In [5]:
# =====================================================
# Cell 3B - IMAGE DATASET TESTING (YOLO format labels)
# Fixed: correct detection + image display + sound only on real falls
# =====================================================
#
# WHY DIFFERENT FROM WEBCAM (Cell 3A):
# Static images have no motion between frames so acceleration
# cannot be computed reliably. Instead we use TWO geometry
# conditions only (Layer 1 + Layer 3):
#   Condition B: torso angle > ANGLE_THRESHOLD  (body tilted)
#   Condition C: hip position > HIP_HEIGHT_RATIO (person is low)
# Sound alert fires only when BOTH B and C are true = real fall.
#
# CONFIG — edit only these lines
DATASET_ROOT   = r'C:\Users\Lenovo ThinkPad X390\OneDrive\Desktop\fall_dataset'
SPLIT          = 'train'    # 'train' or 'val'
DISPLAY_W      = 640        # frame width for display
DISPLAY_EVERY  = 3          # show 1 out of every N frames (1=all, slower)
MODEL_PATH     = 'pose_landmarker_lite.task'
ANGLE_THRESHOLD    = 45     # degrees from vertical — body tilt (Layer 1)
HIP_HEIGHT_RATIO   = 0.60   # hip y / frame height  — low position (Layer 3)
SOUND_COOLDOWN     = 3.0    # seconds between consecutive sound alerts

import pathlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import clear_output

IMAGE_DIR   = pathlib.Path(DATASET_ROOT) / 'images' / SPLIT
LABEL_DIR   = pathlib.Path(DATASET_ROOT) / 'labels' / SPLIT
image_files = sorted(IMAGE_DIR.glob('*.jpg')) + sorted(IMAGE_DIR.glob('*.png'))
print(f'Found {len(image_files)} images in {IMAGE_DIR}')


# ── Helpers ──────────────────────────────────────────────────

def load_yolo_label(label_path):
    """YOLO label: class 0 = fall. Returns True / False / None."""
    if not label_path.exists():
        return None
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if parts and int(parts[0]) == 0:
                return True
    return False


def draw_yolo_box(frame, label_path, w, h):
    """Ground-truth box: orange = fall, green = safe."""
    if not label_path.exists():
        return
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            cls = int(parts[0])
            xc, yc, bw, bh = float(parts[1]), float(parts[2]), \
                              float(parts[3]), float(parts[4])
            x1 = int((xc - bw/2) * w);  y1 = int((yc - bh/2) * h)
            x2 = int((xc + bw/2) * w);  y2 = int((yc + bh/2) * h)
            color = (0, 80, 255) if cls == 0 else (0, 200, 80)
            cv2.rectangle(frame, (x1,y1), (x2,y2), color, 2)
            cv2.putText(frame, 'GT:FALL' if cls==0 else 'GT:SAFE',
                        (x1, max(y1-6,12)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)


def draw_result_badge(frame, detected, ground_truth):
    """TP/FP/FN/TN badge in top-right corner."""
    fh, fw = frame.shape[:2]
    if ground_truth is None:              label, color = 'NO LABEL',        (120,120,120)
    elif detected and ground_truth:       label, color = 'TP Correct Fall',  (0,200,80)
    elif detected and not ground_truth:   label, color = 'FP False Alarm',   (0,140,255)
    elif not detected and ground_truth:   label, color = 'FN Missed Fall',   (0,0,220)
    else:                                 label, color = 'TN Correct Safe',  (80,80,80)
    tw = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.52, 1)[0][0]
    x1, y1 = fw - tw - 24, 10
    cv2.rectangle(frame, (x1,y1), (fw-4, y1+24), color, -1)
    cv2.putText(frame, label, (x1+5, y1+16),
                cv2.FONT_HERSHEY_SIMPLEX, 0.48, (255,255,255), 1, cv2.LINE_AA)


def detect_fall_static(shoulder_mid, hip_mid, frame_h):
    """Fall detection for static images — Layer 1 + Layer 3 only.

    Static images have no motion so Layer 2 (acceleration) cannot
    be used. We use geometry only:
      Condition B: torso angle from vertical > ANGLE_THRESHOLD
      Condition C: hip y-position > HIP_HEIGHT_RATIO x frame_h
    Both must be true = fall detected.
    """
    torso_angle = torso_angle_from_vertical(shoulder_mid, hip_mid)
    hip_ratio   = hip_mid[1] / frame_h
    cond_b = torso_angle > ANGLE_THRESHOLD
    cond_c = hip_ratio   > HIP_HEIGHT_RATIO
    is_fall = cond_b and cond_c
    return is_fall, {
        'accel': 0, 'torso_angle': torso_angle, 'hip_ratio': hip_ratio,
        'cond_a': False, 'cond_b': cond_b, 'cond_c': cond_c, 'fall_count': 0,
    }


def show_frame_inline(frame, diag, is_fall, frame_i, total, tp, fp, fn, tn):
    """Render annotated frame + metrics panel using matplotlib clear_output."""
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4),
                              gridspec_kw={'width_ratios': [2, 1]},
                              facecolor='#111')

    # Left — annotated frame
    axes[0].imshow(rgb)
    axes[0].axis('off')
    axes[0].set_title(
        f'Frame {frame_i} / {total}     '
        f'{"FALL DETECTED" if is_fall else "No Fall"}',
        color='red' if is_fall else 'lightgreen', fontsize=11, pad=6
    )

    # Right — metrics panel
    ax = axes[1]
    ax.set_facecolor('#1a1a1a')
    ax.axis('off')
    total_cm = tp + fp + fn + tn
    acc = (tp+tn)/total_cm*100 if total_cm > 0 else 0
    rows = [
        ('Static Image Mode',                             '#aaa',   10, 'bold'),
        ('(Layer 1 + 3 only)',                            '#666',    8, 'normal'),
        ('',                                              '#000',    4, 'normal'),
        (f'Torso angle: {diag["torso_angle"]:.1f} deg',
         '#aed581' if diag['cond_b'] else '#555',         9, 'normal'),
        (f'  threshold: {ANGLE_THRESHOLD} deg',           '#444',    8, 'normal'),
        (f'Hip height : {diag["hip_ratio"]*100:.1f}%',
         '#ffb74d' if diag['cond_c'] else '#555',         9, 'normal'),
        (f'  threshold: {HIP_HEIGHT_RATIO*100:.0f}%',     '#444',    8, 'normal'),
        ('',                                              '#000',    4, 'normal'),
        (f'[B] angle  {"PASS" if diag["cond_b"] else "fail"}',
         '#aed581' if diag['cond_b'] else '#555',         9, 'normal'),
        (f'[C] height {"PASS" if diag["cond_c"] else "fail"}',
         '#ffb74d' if diag['cond_c'] else '#555',         9, 'normal'),
        ('',                                              '#000',    4, 'normal'),
        (f'TP={tp}  FP={fp}  FN={fn}  TN={tn}',          '#ccc',    8, 'normal'),
        (f'Accuracy: {acc:.1f}%',                         '#fff',   10, 'bold'),
    ]
    y = 0.97
    for text, color, size, weight in rows:
        ax.text(0.05, y, text, color=color, fontsize=size,
                fontweight=weight, transform=ax.transAxes,
                fontfamily='monospace', va='top')
        y -= 0.075

    plt.tight_layout(pad=0.5)
    clear_output(wait=True)
    plt.show()
    plt.close(fig)


# ── MediaPipe IMAGE mode setup ────────────────────────────────
BaseOptions           = mp_python.BaseOptions
PoseLandmarker        = mp_vision.PoseLandmarker
PoseLandmarkerOptions = mp_vision.PoseLandmarkerOptions
VisionRunningMode     = mp_vision.RunningMode

options_img = PoseLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    running_mode=VisionRunningMode.IMAGE,
    num_poses=1,
    min_pose_detection_confidence=0.5,
    min_pose_presence_confidence=0.5,
    min_tracking_confidence=0.5,
)

# ── CSV logger ────────────────────────────────────────────────
ts_str   = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
log_path = f'fall_log_dataset_{ts_str}.csv'
logger   = CSVLogger(log_path)

# ── State ─────────────────────────────────────────────────────
frame_i        = 0
tp = fp = fn = tn = 0
last_sound_t   = 0.0
fall_count     = 0

print('Running...  (Kernel -> Interrupt to stop early)')

# ── Main loop ─────────────────────────────────────────────────
with PoseLandmarker.create_from_options(options_img) as landmarker:
    for img_path in image_files:
        frame = cv2.imread(str(img_path))
        if frame is None:
            continue

        now     = time.time()
        frame_i += 1

        scale = DISPLAY_W / frame.shape[1]
        frame = cv2.resize(frame, (DISPLAY_W, int(frame.shape[0] * scale)))
        h, w  = frame.shape[:2]

        # Ground truth from YOLO label
        label_path   = LABEL_DIR / (img_path.stem + '.txt')
        ground_truth = load_yolo_label(label_path)
        draw_yolo_box(frame, label_path, w, h)

        # MediaPipe pose inference
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB,
                            data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        result = landmarker.detect(mp_image)

        is_fall = False
        diag    = dict(accel=0, torso_angle=0, hip_ratio=0,
                       cond_a=False, cond_b=False, cond_c=False, fall_count=0)

        if result.pose_landmarks:
            lms = result.pose_landmarks[0]

            # LAYER 1 — pixel coords and midpoints
            ls = landmark_point(lms[LEFT_SHOULDER],  w, h)
            rs = landmark_point(lms[RIGHT_SHOULDER], w, h)
            lh = landmark_point(lms[LEFT_HIP],       w, h)
            rh = landmark_point(lms[RIGHT_HIP],      w, h)
            shoulder_mid = (ls + rs) / 2   # midpoint = (p1 + p2) / 2
            hip_mid      = (lh + rh) / 2

            # LAYER 1 + 3 — geometry-only fall detection for static images
            is_fall, diag = detect_fall_static(shoulder_mid, hip_mid, h)
            diag['fall_count'] = fall_count

            # Sound alert — only fires on detected fall, with cooldown
            if is_fall and (now - last_sound_t) > SOUND_COOLDOWN:
                fall_count   += 1
                last_sound_t  = now
                diag['fall_count'] = fall_count
                play_alert_sound()
                print(f'  FALL #{fall_count} detected — {img_path.name}')

            # Skeleton overlay
            draw_skeleton(frame, lms, w, h)
            # Torso axis (cyan)
            cv2.line(frame, tuple(hip_mid.astype(int)),
                     tuple(shoulder_mid.astype(int)), (100, 220, 255), 3)
            # Vertical reference from hip (blue)
            cv2.line(frame, tuple(hip_mid.astype(int)),
                     (int(hip_mid[0]), int(hip_mid[1])-80), (60, 60, 200), 1)

        # Confusion matrix
        if ground_truth is not None:
            if     is_fall and     ground_truth: tp += 1
            elif   is_fall and not ground_truth: fp += 1
            elif not is_fall and   ground_truth: fn += 1
            else:                                tn += 1

        # Draw overlays
        draw_hud(frame, diag, is_fall, 0)
        draw_result_badge(frame, is_fall, ground_truth)
        cv2.putText(frame, img_path.name, (300, 22),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.42, (200,200,200), 1)

        logger.write(now, 0, diag, is_fall)

        # Display every N frames
        if frame_i % DISPLAY_EVERY == 0:
            show_frame_inline(frame, diag, is_fall,
                              frame_i, len(image_files),
                              tp, fp, fn, tn)

logger.close()

# ── Final summary ─────────────────────────────────────────────
total     = tp + fp + fn + tn
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
f1        = 2*precision*recall / (precision+recall) if (precision+recall) > 0 else 0
accuracy  = (tp + tn) / total if total > 0 else 0

clear_output(wait=True)
print(f'\n{"="*44}')
print(f'RESULTS  {frame_i} frames  |  {SPLIT} split')
print(f'{"="*44}')
print(f'  TP (correct falls)  : {tp}')
print(f'  FP (false alarms)   : {fp}')
print(f'  FN (missed falls)   : {fn}')
print(f'  TN (correct safe)   : {tn}')
print(f'{"-"*44}')
print(f'  Accuracy  : {accuracy*100:.1f}%')
print(f'  Precision : {precision*100:.1f}%')
print(f'  Recall    : {recall*100:.1f}%')
print(f'  F1 Score  : {f1*100:.1f}%')
print(f'{"="*44}')
print(f'  Detection method: Angle + Hip Height (static image mode)')
print(f'  Log saved to: {log_path}')


RESULTS  374 frames  |  train split
  TP (correct falls)  : 66
  FP (false alarms)   : 3
  FN (missed falls)   : 147
  TN (correct safe)   : 158
--------------------------------------------
  Accuracy  : 59.9%
  Precision : 95.7%
  Recall    : 31.0%
  F1 Score  : 46.8%
  Detection method: Angle + Hip Height (static image mode)
  Log saved to: fall_log_dataset_20260524_142843.csv


In [8]:
# Cell 4 - Post-session review (run after stopping Cell 3)
import pandas as pd

df = pd.read_csv(log_path)
print(f'Frames   : {len(df)}')
print(f'Falls    : {df["fall_detected"].sum()}')
print(f'Duration : {float(df["timestamp"].iloc[-1]) - float(df["timestamp"].iloc[0]):.1f}s')
print(f'Avg FPS  : {df["fps"].mean():.1f}')
display(df[df['fall_detected']==1][['timestamp','accel','torso_angle','hip_ratio']].head(20))

df['t_rel'] = df['timestamp'].astype(float) - float(df['timestamp'].iloc[0])
fall_times  = df[df['fall_detected']==1]['t_rel'].values

fig, axes = plt.subplots(3, 1, figsize=(14, 7), facecolor='#111', sharex=True)
specs = [
    ('Vertical Acceleration (px/s2)', 'accel',       '#4fc3f7', 800),
    ('Torso Angle (deg)',             'torso_angle',  '#aed581',  45),
    ('Hip Height Ratio',             'hip_ratio',    '#ffb74d', 0.65),
]
for ax, (title, col, colour, thr) in zip(axes, specs):
    ax.set_facecolor('#1a1a1a')
    ax.tick_params(colors='#888')
    for sp in ax.spines.values():
        sp.set_color('#333')
    ax.plot(df['t_rel'], df[col], color=colour, linewidth=0.8)
    ax.axhline(thr, color='#ef5350', linewidth=0.8, linestyle='--')
    for ft in fall_times:
        ax.axvline(ft, color='red', alpha=0.5, linewidth=1)
    ax.set_title(title, color='#aaa', fontsize=9)
    ax.set_ylabel(col, color='#888', fontsize=8)
axes[-1].set_xlabel('Time (s)', color='#888')
fig.suptitle(f'Session Review - {log_path}', color='#ccc', fontsize=11)
plt.tight_layout()
out_png = log_path.replace('.csv', '_review.png')
plt.savefig(out_png, dpi=120, facecolor='#111')
plt.show()
print(f'Saved to {out_png}')

ModuleNotFoundError: No module named 'pandas'

## Tuning guide
| Parameter | Location | Effect |
|---|---|---|
| `ACCEL_THRESHOLD = 800` | `FallDetector` | Lower = more sensitive to fast motion |
| `ANGLE_THRESHOLD = 45` | `FallDetector` | Tilt angle before condition B fires |
| `HIP_HEIGHT_RATIO = 0.65` | `FallDetector` | Adjust for your camera mounting height |
| `COOLDOWN = 5.0` | `FallDetector` | Min seconds between consecutive alerts |
| `SOURCE` | Cell 3 | `'webcam'` or a file path like `'test.mp4'` |
| `CAMERA_INDEX` | Cell 3 | 0 = built-in, 1+ = external webcam |
| `MODEL_PATH` | Cell 3 | Swap `_lite` for `_full` for higher accuracy (larger model) |

## MediaPipe API change reference (0.9 vs 0.10)
| Old `mp.solutions` (0.9) | New `mediapipe.tasks` (0.10+) |
|---|---|
| `mp.solutions.pose.Pose()` | `PoseLandmarker.create_from_options(options)` |
| `pose.process(rgb_frame)` | `landmarker.detect_for_video(mp_image, timestamp_ms)` |
| `results.pose_landmarks.landmark[i]` | `result.pose_landmarks[0][i]` |
| `mp.solutions.drawing_utils.draw_landmarks(...)` | Manual `cv2.line` / `cv2.circle` |
| `mp.solutions.pose.PoseLandmark.LEFT_HIP` | Integer constant `23` |